# TD2 — Classification: classic ML vs. zero-shot LLM

**~1h30 · the second brick of the PIM assistant you build across TD1–TD5.**

A core PIM job for a distributor: **file each incoming product under the right leaf category** of the catalog taxonomy (`../data/taxonomy.json`, 22 leaves). Do it by hand for thousands of supplier products and you drown; this is exactly the chore your agent will automate in TD5. Here you'll solve that *one* task **two ways** and feel the trade-offs that drove the whole ML → GenAI shift:

- **Approach A — classic ML:** embeddings (from TD1) as features + a **logistic regression**.
- **Approach B — zero-shot LLM** (Claude **Haiku**), in three escalating variants:
  - **B.1** naive free-text — ask, in plain language, which category the product belongs to;
  - **B.2** **structured output** (Pydantic) — constrain the answer to the 22 valid leaves;
  - **B.3** **Pydantic + reasoning-first** — let the model think before it commits (shown as a code *pattern*, **not run** here — see §5).

Then we **compare** the measured approaches (A / B.1 / B.2) and reflect on the thing that made all of this measurable: a **golden dataset**.

**How to work:** run the notebook top to bottom; complete the `# TODO` cells; **write your own analysis answers** (you may use Claude Code to help you *think*, but the words must be yours); ask Claude Code first whenever you're stuck. Each `# TODO` ends with a quick **self-check** (`assert …`) so you can confirm your code is right **without any solution**.

> ⚠️ Approaches **B.1 and B.2** make **real Haiku API calls** (~80 total, on the ~40-item test set). **B.3 is markdown-only — zero API calls.** You need `ANTHROPIC_API_KEY` in a `.env` file at the project root. We keep `max_tokens` small and only evaluate on the test set.
>
> Because the LLM is **nondeterministic**, your exact numbers will differ a little from a neighbour's or from a re-run — that's expected, and itself part of the lesson (§7). 🚀

## 1. Setup

In [ ]:
import json
import os
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import anthropic

# Shared PIM catalog — the same dataset used in TD1, TD3, TD4.
df = pd.read_csv("../data/products.csv")

# The text we classify for each product: its name plus the long description (same as TD1).
df["text"] = df["name"] + " — " + df["long_description"]

# The 22 leaf categories = the valid labels.
with open("../data/taxonomy.json") as f:
    taxo = json.load(f)
LEAVES = [sub["name"] for top in taxo["categories"] for sub in top["subcategories"]]
LEAF_SET = set(LEAVES)
assert len(LEAVES) == 22, f"expected 22 leaf categories, got {len(LEAVES)}"

print(f"Loaded {len(df)} products | {len(LEAVES)} leaf categories")
print("Leaves:", LEAVES)

In [ ]:
# Stratified train/test split (fixed seed → reproducible): every leaf is in both halves.
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=0, stratify=df["category"]
)

X_train_text = train_df["text"].tolist()
X_test_text = test_df["text"].tolist()
y_train = train_df["category"].tolist()
y_test = test_df["category"].tolist()

print(f"Train: {len(train_df)} products | Test: {len(test_df)} products "
      f"(~{len(test_df) / len(LEAVES):.1f} per leaf)")
print("All 22 leaves present in the test set:",
      test_df["category"].nunique() == len(LEAVES))

In [ ]:
# LLM client (Haiku only) + local embedding model (free, from TD1).
load_dotenv()  # reads ANTHROPIC_API_KEY from .env at the project root
if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY not found. Create a .env file at the project root with\n"
        "ANTHROPIC_API_KEY=sk-ant-...  (see resources/setup_guide.md)."
    )
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Client + embedding model ready.")

## 1b. Shared helpers

Two pre-built helpers do the plumbing so each approach below is just its conceptual core:

- **`evaluate(y_true, y_pred)`** — prints overall accuracy and how often the two *confusable* leaf pairs (*Headphones* ↔ *Wireless Earbuds*, *Smartwatches* ↔ *Fitness Trackers*) get swapped, then returns the accuracy.
- **`run_over_testset(classify_fn)`** — loops your per-item classifier over the test texts, times it, and accumulates token usage. Each approach only has to supply a `classify_fn(text) -> (label, usage)`.

In [ ]:
CONFUSABLE_PAIRS = [
    ("Headphones", "Wireless Earbuds"),
    ("Smartwatches", "Fitness Trackers"),
]


def evaluate(y_true, y_pred):
    """Score predictions against the gold labels. Pre-built -- nothing to fill in.

    Prints, then returns, the overall accuracy = fraction of test items whose
    predicted leaf equals the true leaf.

    It also prints, for each CONFUSABLE_PAIR, the number of *swaps*: items whose
    true category is one leaf of the pair but that got predicted as the OTHER leaf
    of the same pair (e.g. a true 'Headphones' predicted as 'Wireless Earbuds', or
    vice-versa). These near-duplicate leaves share vocabulary, so swaps between them
    are the most common, most instructive errors -- watch whether each approach
    reduces them.

    Args:  y_true, y_pred -- equal-length sequences of leaf names
           (a None in y_pred, from an unparseable answer, simply counts as wrong).
    Returns: accuracy as a float in [0, 1].
    """
    y_true, y_pred = list(y_true), list(y_pred)
    correct = sum(t == p for t, p in zip(y_true, y_pred))
    acc = correct / len(y_true)
    print(f"Accuracy: {acc:.1%}  ({correct}/{len(y_true)})")
    for a, b in CONFUSABLE_PAIRS:
        swaps = sum(
            (t == a and p == b) or (t == b and p == a)
            for t, p in zip(y_true, y_pred)
        )
        n_pair = sum(t in (a, b) for t in y_true)
        print(f"  {a} <-> {b}: {swaps} swap(s) among {n_pair} test items")
    return acc


def run_over_testset(classify_fn):
    """Run one per-item classifier over the whole test set. Pre-built -- nothing to fill in.

    This is the shared driver every approach below plugs into: you only ever write a
    `classify_fn(text) -> (label, usage)` for a SINGLE product, and this function
    loops it over all the test texts, measures wall-clock time, and adds up token usage.

    Args:
      classify_fn -- a function taking one product text and returning a tuple
        (label, usage), where `usage` is either a dict with integer keys
        'input_tokens' and 'output_tokens', or None when no API call was made
        (e.g. the classic-ML approach).
    Returns:
      preds        -- list of predicted labels, one per test item, in test order;
      avg_latency  -- mean seconds per item (total elapsed / number of items);
      usage_totals -- dict summing 'input_tokens' and 'output_tokens' over all calls.
    """
    preds = []
    usage_totals = {"input_tokens": 0, "output_tokens": 0}
    t0 = time.perf_counter()
    for text in X_test_text:
        label, usage = classify_fn(text)
        preds.append(label)
        if usage is not None:
            usage_totals["input_tokens"] += usage["input_tokens"]
            usage_totals["output_tokens"] += usage["output_tokens"]
    dt = time.perf_counter() - t0
    avg_latency = dt / len(X_test_text)
    return preds, avg_latency, usage_totals

# Collect every approach's metrics here for the final comparison table.
results = {}

## 2. Approach A — classic ML (embeddings + logistic regression)

The classic recipe: turn each product text into a fixed-length **embedding** (the MiniLM vectors from TD1), then train a **logistic regression** to map those vectors to one of the 22 leaves — the everyday PIM chore of *filing* each product.

We pre-embed the train and test texts for you; you write and evaluate the classifier — then we'll read off what this approach is good and bad at.

In [ ]:
# Pre-built: embed train + test texts with MiniLM (local, free).
X_train = embed_model.encode(X_train_text, show_progress_bar=False)
X_test = embed_model.encode(X_test_text, show_progress_bar=False)
print("Feature matrices:", X_train.shape, X_test.shape)  # expected ~ (158, 384) (40, 384)

In [ ]:
# TODO: train and evaluate the classic-ML classifier.
#   1. fit a LogisticRegression(max_iter=1000) on (X_train, y_train)  -> `clf`
#   2. predict on X_test                                              -> `pred_a`
#   3. score it by calling evaluate(y_test, pred_a)                   -> `acc_a`
# (Instruction only — no solution here. The self-check tells you if it worked.)
clf = None      # TODO
pred_a = None   # TODO
acc_a = None    # TODO

# Self-check (passes once your code is correct):
assert pred_a is not None and len(pred_a) == len(y_test), "pred_a not built yet"
assert set(pred_a) <= LEAF_SET, "predictions must all be valid leaves"

In [ ]:
# Pre-built: record Approach A's metrics for the final comparison table.
results["A — LogReg"] = {
    "accuracy": acc_a,
    "pct_valid": 1.0,          # sklearn can only output the 22 trained labels
    "input_tokens": 0,         # no API calls
    "output_tokens": 0,
    "avg_latency_s": None,     # ~instant, no network
    "needs_labels": "Yes",
    "explainability": "coefficients",
    "deterministic": "Yes",
}
print("Recorded Approach A.")

**Takeaway.** Once trained, Approach A is **cheap, instant, deterministic and explainable** (you can inspect the learned coefficients) — but it **needed a labeled training set**, and its errors cluster on the **near-duplicate leaves** whose vocabularies overlap: *Headphones* ↔ *Wireless Earbuds*, *Smartwatches* ↔ *Fitness Trackers* — exactly the clusters that touched in TD1's 2D map. The upside of having labels: you could attack those *systematically* (more data for the confused leaves, a stronger embedding, class weighting) and re-measure. Hold that thought for §7.

## 3. Approach B.1 — zero-shot LLM, naive free-text

Now the GenAI way: **no training at all** — handy when a new distributor catalog arrives and you have no labels yet. We list the 22 leaf names in the prompt and ask Haiku, in plain language, **which category the product belongs to** — and leave it free to answer however it likes.

You'll write two things: `classify_naive` (the API call with that loose prompt), and — because the model is free to reply in a full sentence — a **lenient parser** `extract_label` that tries to recover a single leaf name from whatever text comes back. Then we run both over the test set and see how well that holds up.

In [ ]:
def classify_naive(text):
    """Zero-shot, free-text. Return (predicted_label: str, usage: dict).
    Build a LOOSE prompt that lists LEAVES and the product and asks an OPEN
    question ("Which category does this product belong to?") -- do NOT tell the
    model to reply with only the label. Call client.messages.create(model=MODEL,
    max_tokens=128, ...) and return:
      - the RAW stripped text of the first content block (the model often answers
        in a full sentence -- that's the point), and
      - a usage dict {'input_tokens': ..., 'output_tokens': ...}.
    """
    # TODO: implement
    raise NotImplementedError

# Self-check once implemented (makes 1 API call):
label0, usage0 = classify_naive(X_test_text[0])
print(repr(label0))
assert isinstance(label0, str) and usage0["input_tokens"] > 0

Run the smoke-test above and look at what comes back. Nothing **constrained** the model: we asked an open question, so Haiku is free to answer in whatever form it likes — a bare label, a full sentence (*"This product belongs to the Headphones category."*), a hedge naming two categories, or a synonym that isn't one of our 22 leaves at all.

That raw text isn't directly comparable to a gold label. To turn it into a verdict we need a step that asks: **which of the 22 leaf names actually appears in the answer?** If exactly one leaf name is present, that's our prediction; if none or several are, the answer is unusable. That's the job of the `extract_label` parser you write next.

In [ ]:
def extract_label(text, leaves):
    """Map a raw free-text answer to a SINGLE leaf, leniently.
    Scan `leaves` for any whose name appears in `text`, case-insensitively
    (a simple substring test). Return that leaf if EXACTLY ONE matches; return
    None if none match (synonym / singular / pure prose) or if several match
    (ambiguous -- the model named more than one category).
    """
    # TODO: implement (a couple of lines; handle the 0-match and >1-match cases)
    raise NotImplementedError

# Self-check once implemented:
assert extract_label("This is clearly in the Headphones category.", LEAVES) == "Headphones"
assert extract_label("Could be Headphones or Wireless Earbuds.", LEAVES) is None  # ambiguous
assert extract_label("These earphones sound great.", LEAVES) is None              # synonym, no leaf
print("extract_label self-check passed.")

> **Predict first (10 seconds — just for you, not graded).** Before you run the batch below: **(a)** will Haiku usually pick the *right* category? **(b)** will a simple parser *always* be able to recover exactly one of the 22 leaves from its answer? Jot a guess, then check it against the numbers.

In [ ]:
# Pre-built: run the naive classifier over the test set (raw free-text answers),
# then parse each one leniently into a leaf with YOUR extract_label.
raw_b1, lat_b1, usage_b1 = run_over_testset(classify_naive)
pred_b1 = [extract_label(raw, LEAVES) for raw in raw_b1]

# 'Invalid' = the lenient parser could not recover exactly one leaf (None).
invalid_idx = [i for i, p in enumerate(pred_b1) if p is None]
print(f"Unparseable / ambiguous answers (extract_label -> None): "
      f"{len(invalid_idx)}/{len(pred_b1)}")
print("\nExamples of raw answers that failed to parse cleanly:")
for i in invalid_idx[:5]:
    print(f"  - {raw_b1[i]!r}")

# Accuracy over ALL test items: None (unparseable) counts as a wrong prediction.
acc_b1 = evaluate(y_test, pred_b1)
results["B.1 — naive"] = {
    "accuracy": acc_b1,
    "pct_valid": (len(pred_b1) - len(invalid_idx)) / len(pred_b1),
    "input_tokens": usage_b1["input_tokens"],
    "output_tokens": usage_b1["output_tokens"],
    "avg_latency_s": lat_b1,
    "needs_labels": "No",
    "explainability": "none",
    "deterministic": "No",
}

**Takeaway.** Haiku almost always *understood* the product — but turning its free-text answer into one of the 22 leaves forced us to hand-write a **brittle parser**, and a fraction of answers still came back **unparseable or ambiguous** (a synonym like "earphones", a singular form, or *two* categories named). The model's understanding is fine; the **free-text interface** is what's unreliable for an automated PIM — those `None`s are silent failures. The fix isn't a cleverer parser: it's to stop parsing and **constrain the output** so every answer is a valid leaf by construction. That's B.2.

## 4. Approach B.2 — structured output with Pydantic

The naive interface was unreliable because we let the model choose the *form* of its answer. The fix is to **declare the shape we want and have the SDK enforce it.**

**Pydantic** is the standard Python library for declaring typed data models: you write a class whose fields have types, and Pydantic validates that incoming data matches them. The Anthropic SDK plugs into it directly through `client.messages.parse(..., output_format=YourModel)` — the SDK turns your Pydantic model into a JSON schema, constrains the model to emit JSON matching that schema, and validates the reply straight into a typed Python object you read off `resp.parsed_output`. No hand-written parser, no `None`s.

Here the model needs a single field, `category`, **constrained to the 22 leaves** (a `Literal`), so every answer is a valid leaf *by construction*. You write the classifier; then we run it over the test set and compare what changed versus the naive version — both its **validity** and its **accuracy**.

In [ ]:
def classify_structured(text):
    """Pydantic-constrained classification. Return (predicted_label: str, usage: dict).
    Steps:
      1. define a Pydantic BaseModel with a SINGLE field `category` typed as
         Literal[tuple(LEAVES)] -- this constrains it to exactly the 22 leaf names;
      2. call client.messages.parse(model=MODEL, max_tokens=64, messages=[...],
         output_format=YourModel) with a short 'classify this product' prompt;
      3. read the validated, typed result from resp.parsed_output -> .category;
      4. return (label, {'input_tokens': ..., 'output_tokens': ...}) from resp.usage.
    (Instruction only -- no solution here. The self-check tells you if it worked.)
    """
    # TODO: implement
    raise NotImplementedError

# Self-check once implemented (makes 1 API call):
label0, usage0 = classify_structured(X_test_text[0])
print(repr(label0))
assert label0 in LEAF_SET

In [ ]:
# Pre-built: run over the test set; confirm 100% valid; evaluate.
pred_b2, lat_b2, usage_b2 = run_over_testset(classify_structured)

n_valid_b2 = sum(p in LEAF_SET for p in pred_b2)
print(f"Valid outputs: {n_valid_b2}/{len(pred_b2)} (should be 100% by construction)")

acc_b2 = evaluate(y_test, pred_b2)
results["B.2 — enum"] = {
    "accuracy": acc_b2,
    "pct_valid": n_valid_b2 / len(pred_b2),
    "input_tokens": usage_b2["input_tokens"],
    "output_tokens": usage_b2["output_tokens"],
    "avg_latency_s": lat_b2,
    "needs_labels": "No",
    "explainability": "none",
    "deterministic": "No",
}

**Question.** Compare B.2 to B.1 using *your* two runs. Did **accuracy** change much? Did **validity** (% of answers that are a usable leaf) change? Structured output fixed *one* of those two things far more than the other — say clearly which, and why that distinction matters for an automated pipeline.

**Your answer** — write it yourself. You may use AI to help you think it through, but the written answer must be in your own words.

_..._

## 5. Approach B.3 — reasoning-first *(concept only — not run here)*

There's a well-known refinement on top of B.2: let the model **think before it commits**. You reuse the exact same `client.messages.parse` + Pydantic machinery from §4 — but add a field, and **field order matters.** The model emits fields in the order you declare them, so if you put **`reasoning` FIRST, then `category`**, it must write its reasoning *before* it picks the category — a lightweight **chain-of-thought**. Forcing it to weigh the distinguishing feature (over-ear vs in-ear, standalone watch vs fitness band) *before* deciding usually nudges accuracy up on exactly the hard, confusable cases — and you get a human-readable justification for free:

```python
from pydantic import BaseModel
from typing import Literal

class Categorization(BaseModel):
    reasoning: str                    # MUST come first -> the model reasons before committing
    category: Literal[tuple(LEAVES)]  # then commits to one of the 22 leaves

resp = client.messages.parse(
    model="claude-haiku-4-5",
    max_tokens=512,                   # room for the reasoning text
    messages=[{"role": "user",
               "content": f"Classify this product into one catalog category. "
                          f"First reason briefly, then give the category.\n\nProduct: {text}"}],
    output_format=Categorization,
)
result = resp.parsed_output           # -> result.reasoning, result.category
```

**We do not run B.3 in this notebook**, on purpose — to keep the API budget low (B.1 + B.2 already spend ~80 calls on the test set, and B.3 would add ~40 more with a much larger `max_tokens`). On our clean, well-separated 22-class taxonomy the gain over B.2 is **small and noisy** anyway (see §7: on a ~40-item test set a one- or two-item swing is within run-to-run variation). It stays in the toolbox as a **reusable technique** — reasoning-first structured output is exactly what your agent will lean on in TD5 when accuracy on hard cases actually matters. (Want to see it in action? It's the first *Going further* task.)

## 6. Comparison

Time to put the **three measured approaches (A / B.1 / B.2)** side by side. We've collected each one's metrics in `results`; you'll write the **cost** helper and then assemble the table.

For **cost**, Haiku is roughly **\$1 / million input tokens** and **\$5 / million output tokens**; we report the cost **per 1000 classifications** (extrapolated from the test-set usage) in euros (≈ 0.92 €/\$).

> **Predict first:** which approach will be **cheapest**? Which **most accurate**? Which would you trust to run unattended on thousands of products? Then check the table.

In [ ]:
# Pricing constants (Haiku ~$1/Mtok in, $5/Mtok out) + a euro rate. (given)
PRICE_IN_PER_MTOK = 1.0
PRICE_OUT_PER_MTOK = 5.0
USD_TO_EUR = 0.92
N_TEST = len(X_test_text)

def cost_eur_per_1000(input_tokens, output_tokens):
    """Cost in EUR per 1000 classifications, extrapolated from the test-set usage.
    Steps:
      - if BOTH token counts are 0, return 0.0 (no API calls were made);
      - dollars for the whole test set =
            input_tokens  * PRICE_IN_PER_MTOK  / 1e6
          + output_tokens * PRICE_OUT_PER_MTOK / 1e6;
      - divide by N_TEST -> $/item;  * 1000 -> $/1000;  * USD_TO_EUR -> EUR/1000.
    """
    # TODO: implement using the constants above
    raise NotImplementedError

# Self-check once implemented:
assert cost_eur_per_1000(0, 0) == 0.0
assert cost_eur_per_1000(1_000_000, 0) > 0
print("cost_eur_per_1000 self-check passed.")

In [ ]:
# Pre-built: assemble the comparison table over the 3 measured approaches in `results`
# (A / B.1 / B.2 — B.3 was not run, so it isn't in `results`). Your cost_eur_per_1000
# above feeds the cost column; everything else was recorded as each approach ran.
rows = []
for name, r in results.items():
    rows.append({
        "approach": name,
        "accuracy": round(r["accuracy"], 3),
        "pct_valid": round(r["pct_valid"], 3),
        "cost_eur_/1000": round(
            cost_eur_per_1000(r["input_tokens"], r["output_tokens"]), 4),
        "avg_latency_s": (None if r["avg_latency_s"] is None
                          else round(r["avg_latency_s"], 3)),
        "needs_labels": r["needs_labels"],
        "explainability": r["explainability"],
        "deterministic": r["deterministic"],
    })
comparison = pd.DataFrame(rows).set_index("approach")
comparison

> **Note on B.3.** **B.3 (Pydantic + reasoning-first)** isn't a row in this table because we didn't run it (§5). It's a known refinement *on top of* B.2: same always-valid output, plus a reasoning step that can lift accuracy on the hard, confusable leaves — at the price of more **output tokens** (cost) and higher **latency** (bigger `max_tokens`, longer generation). On this clean 22-class task with a ~40-item test set the expected gain is small and noisy, so we left it un-benchmarked to save API budget. When accuracy on ambiguous cases is the real bottleneck, you'd benchmark it on *your* golden set and decide if the extra cost is worth it.

**Question.** Based on *your* table, which approach would you recommend for the PIM's auto-categorization feature — and under what constraints would your choice change (e.g. no labeled data yet, a hard accuracy target, a tight cost or latency budget, a need for determinism / auditability)?

**Your answer** — write it yourself. You may use AI to help you think it through, but the written answer must be in your own words.

_..._

## 7. What made this measurable — and why generation is harder

Step back: **why were we able to put numbers on any of this?** Because we had a **golden dataset** — inputs paired with known-correct labels (our held-out **~40-item test set, ~2 per leaf**, from §1). Every accuracy figure, every comparison, every "would B.3 beat B.2?" *could only be answered* against that reference set. Classification is lucky: someone already labeled the catalog, so the golden dataset comes for free.

**Generative GenAI usually isn't so lucky.** When a system *writes* a product description or *answers* a catalog question (TD3's RAG assistant), there is **no single correct output** — so you can't just compute accuracy. To measure and improve such a system you must first **build** a golden dataset: curate representative inputs (e.g. real user queries), pair each with a reference (an expected answer, a set of required facts, or the ids of the relevant documents), label them with humans (optionally AI-assisted then human-reviewed), and keep the set small, representative and held-out. That curation is often the **hardest and most valuable part** of a GenAI project — and the prerequisite for any systematic iteration.

**A caveat even here:** our test set is *tiny* — ~40 items, ~2 per leaf — so a single misclassification swings a per-class number a lot. That's also why your numbers wobble between runs. Representativeness and size matter; small golden sets give noisy verdicts.

**On *improving* a model.** With a golden dataset, the **ML side** iterates systematically: error analysis → more data / better features / hyperparameters → retrain → re-measure, all offline and deterministic. The **LLM side** is **prompt engineering** — closer to an art: every eval run costs API calls and latency, nondeterminism makes small gains noisy, and it's black-box (the `reasoning` helps but doesn't guarantee). You can't "train" it on your labels without **fine-tuning** (a lecture topic). Open-ended grading — **LLM-as-a-judge** — is explained in TD3.

**Question.** In TD3 you'll build a RAG assistant that *writes* product descriptions and *answers* catalog questions — tasks with no single correct output, so the test set we used here doesn't exist. What would a **golden dataset** for that assistant look like (what's an *input*, what's the *reference*)? How would you build it, and what makes that genuinely hard?

**Your answer** — write it yourself. You may use AI to help you think it through, but the written answer must be in your own words.

_..._

## 🚀 Going further (optional — for when you've finished the core)

No solutions for these — use **Claude Code** as your pair-programmer. Pick whatever interests you:

- **Actually run B.3.** Implement the §5 pattern (`messages.parse` + the `Categorization` model) and run it *only on the handful of items where A and B.2 disagree*. Read the `reasoning` strings — are they sound? Did reasoning-first flip any to correct? (Stay on the small subset to keep the cost tiny.)
- **Better features.** Re-run Approach A with a stronger embedding (`all-mpnet-base-v2`) instead of MiniLM. Does it separate *Headphones* vs *Wireless Earbuds* any better? At what cost in speed/size?
- **See the mistakes.** Build a confusion matrix for B.2 (`sklearn.metrics.confusion_matrix` + a heatmap). Which leaves bleed into which?
- **Break the prompt.** Make the B.2 prompt vague or misleading and watch accuracy move while validity stays 100% — proof that structure guarantees the *form* of the answer, not its *correctness*.

## 8. Wrap-up

What you take away from TD2:

- **Same task, two paradigms.** Classic ML (embeddings + LogReg) needs **labeled data + training** but is cheap, fast, deterministic and explainable. The zero-shot **LLM** needs *no* training, but costs money and latency per call, is nondeterministic, and is a black box.
- **Structured output is non-negotiable** for any automated LLM pipeline: it fixes **validity** (B.2). **Reasoning-first** (B.3) can nudge **accuracy** and gives a readable explanation — but proves nothing on its own.
- **Pick by constraints:** do you have labels? a hard accuracy target? a cost / latency / determinism / explainability requirement? There's no universally best answer.
- **Everything rested on a golden dataset.** Classification had one for free; generative GenAI usually doesn't, and building one is the hard, valuable prerequisite for measuring and improving.

**Next:** **TD3** turns your TD1 search into a real **RAG** with ChromaDB and uses this assistant to *generate* — where there's no ground truth, so you'll build a golden set and lean on **judging**. The categorization technique you built here is the same step the **agent reuses in TD5**. ✅